In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
import matplotlib.pyplot as plt
import dask.dataframe as dd
from dask.diagnostics import ProgressBar
import pyarrow as pa
import ast
import swifter
THRESHOLD_TIMESTAMPS = 16

In [ ]:
df = dd.read_parquet("merged_dataset.parquet")
print(len(df))

In [ ]:
list_cols_data_set = ["page_visit_timestamp", "add_to_cart_timestamp", "product_buy_timestamp",
                      "remove_from_cart_timestamp", "search_query_timestamp"] 

In [ ]:
df[list_cols_data_set] = df[list_cols_data_set].map_partitions(
    lambda pdf: pdf.apply(
        lambda col: col.apply(
            lambda x: x.tolist() if isinstance(x, np.ndarray) else ([] if x is None else x)
        )
    )
)

In [ ]:
df["Total_timestamps"] = df[list_cols_data_set].map_partitions(
    lambda pdf: pdf.apply(
        lambda row: sum(len(x) if isinstance(x, list) else 0 for x in row),
        axis=1
    )
)

In [ ]:
with ProgressBar():
    passed_lenghts_timestamps = df[df["Total_timestamps"] > THRESHOLD_TIMESTAMPS].compute()
    print(f"Total after the threshold {len(passed_lenghts_timestamps)}")
    #print(f"Median after the threshold {np.median(passed_lenghts_timestamps)}")


In [ ]:
print(passed_lenghts_timestamps.head())

In [ ]:
def convert_list_to_unix(ls):
    return [int(datetime.strptime(x, "%Y-%m-%d %H:%M:%S").timestamp()) for x in ls]


In [ ]:
with ProgressBar():
    passed_lenghts_timestamps["page_visit_timestamp_unix"] = (
        passed_lenghts_timestamps["page_visit_timestamp"].swifter.apply(convert_list_to_unix)
    )

    passed_lenghts_timestamps["add_to_cart_timestamp_unix"] = (
        passed_lenghts_timestamps["add_to_cart_timestamp"].swifter.apply(convert_list_to_unix)
    )

    passed_lenghts_timestamps["product_buy_timestamp_unix"] = (
        passed_lenghts_timestamps["product_buy_timestamp"].swifter.apply(convert_list_to_unix)
    )

    passed_lenghts_timestamps["remove_from_cart_timestamp_unix"] = (
        passed_lenghts_timestamps["remove_from_cart_timestamp"].swifter.apply(convert_list_to_unix)
    )

    passed_lenghts_timestamps["search_query_timestamp_unix"] = (
        passed_lenghts_timestamps["search_query_timestamp"].swifter.apply(convert_list_to_unix)
    )


In [ ]:
def merge_unix_timestaps_per_user(row):
    merge_row = []
    for col in [
        "page_visit_timestamp_unix",
        "add_to_cart_timestamp_unix",
        "product_buy_timestamp_unix",
        "remove_from_cart_timestamp_unix",
        "search_query_timestamp_unix" 
    ]:
        if isinstance(row[col], list):
            merge_row.extend(row[col])
    return merge_row
        

In [ ]:
passed_lenghts_timestamps["TS_All"] = passed_lenghts_timestamps.apply(merge_unix_timestaps_per_user, axis=1)



In [ ]:
print(passed_lenghts_timestamps["TS_All"])

In [ ]:
passed_lenghts_timestamps.to_parquet("TsAllParquet.parquet")

In [ ]:
ps = pd.read_parquet("TsAllParquet.parquet")


In [ ]:
print(ps["TS_All"].head())

In [ ]:
avg_days = []

for (session_id, timestamps) in enumerate(ps["TS_All"]):
    ts = [int(x) for x in timestamps]   # ensure ints

    min_ts = min(ts)
    max_ts = max(ts)
    avg_ts = sum(ts) / len(ts)

    diff_seconds = max_ts - min_ts
    avg_days.append(diff_seconds)

    diff = datetime.timedelta(seconds=diff_seconds)

    """
    print(f"This is the Len of the timestamps in this sessions {len(ts)}")
    print(f"This is the min TimeStamp of this Session {session_id}: {datetime.datetime.fromtimestamp(min_ts).strftime('%d-%m-%Y %H:%M:%S')}")

    print(f"This is the max TimeStamp of this Session {session_id} {datetime.datetime.fromtimestamp(max_ts).strftime('%d-%m-%Y %H:%M:%S')}")

    print(f"This is the avg TimeStamp of this Session {session_id}: {datetime.datetime.fromtimestamp(avg_ts).strftime('%d-%m-%Y %H:%M:%S')}")

    print(f"Duration of session: {diff}")
    print("=======================================================================")
    """
    """if session_id == 200:
        break
    """

In [ ]:
converted = [datetime.timedelta(seconds=x) for x in avg_days]
print(f"Total Avarage per days of all the sessions {np.mean(converted)})